# manga-lens: fine-tune nhận dạng nhân vật (face / body)
Bật trước: **Internet On**, **GPU T4**, Secret `HF_TOKEN` (Attached). Mỗi phiên mới phải chạy lại từ đầu.
Quy trình: train trên 87 bộ **train**, chọn epoch/cấu hình trên **val**, rồi chạy **test đúng một lần** cho cấu hình đã chọn.


In [ ]:
!pip -q install huggingface_hub
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 1. Tải Manga109

In [ ]:
!hf download hal-utokyo/Manga109 Manga109_released_2026_05_21.zip --repo-type dataset --local-dir /kaggle/working/m109
!cd /kaggle/working/m109 && unzip -q Manga109_released_2026_05_21.zip && rm Manga109_released_2026_05_21.zip
!ls /kaggle/working/m109

## 2. Code (clone từ GitHub) và nhãn nhân vật
Nếu `ls` cho thấy `images/` và `annotations/` ở thư mục con khác, sửa `ROOT`. Cần đã push các script mới (`extract_characters.py`, `eval_cluster.py`, `train_reid.py`).

In [ ]:
!git clone https://github.com/CPhuongg/manga-lens.git

In [ ]:
import sys, os
sys.path.insert(0, "manga-lens/scripts")
from pathlib import Path
from convert_manga109 import split_titles   # cùng cách chia (seed 0, theo bộ) như lúc train detector
ROOT = "/kaggle/working/m109/Manga109_released_2026_05_21"
titles = [p.stem for p in Path(ROOT, "annotations").glob("*.xml")]
sp = split_titles(titles, seed=0)
os.makedirs("/kaggle/working/titles", exist_ok=True)
for s, t in sp.items():
    Path(f"/kaggle/working/titles/{s}_titles.txt").write_text(chr(10).join(sorted(t)))
    print(s, len(t), "bộ")
!cd manga-lens && PYTHONUTF8=1 python scripts/extract_characters.py --root {ROOT} --yolo /kaggle/working/titles --out /kaggle/working/characters.json

## 3. Thử nhanh (2 bộ, vài bước) để kiểm tra pipeline

In [ ]:
CACHE = "/kaggle/temp/crops" if os.path.isdir("/kaggle/temp") else "/kaggle/working/crops"
COMMON = f"--root {ROOT} --chars /kaggle/working/characters.json --cache {CACHE}_smoke --out /kaggle/working/out/smoke"
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag smoke --epochs 1 --iters 10 --max-books 2

## 4. Face: so vài cấu hình, chọn trên VAL
Mỗi cấu hình ~10 epoch. `CACHE` được dùng chung nên crop chỉ cắt một lần. Số theo dõi mỗi epoch là retrieval trên val;
số quyết định là `eval_cluster --no-test` ở bước 5.

In [ ]:
COMMON = f"--root {ROOT} --chars /kaggle/working/characters.json --cache {CACHE} --out /kaggle/working/out/reid"
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_a --unfreeze 4  --lr-bb 2e-5 --epochs 10

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_b --unfreeze 12 --lr-bb 1e-5 --epochs 10

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_c --unfreeze 4  --lr-bb 2e-5 --temp 0.05 --epochs 10

## 5. Chọn trên VAL (không đụng test)

In [ ]:
for tag in ["f_a", "f_b", "f_c"]:
    print("=====", tag)
    !cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag {tag} --feat face --emb-dir /kaggle/working/out/reid --no-test | grep -E "feat=|thr|chọn"

## 6. Test — một lần, cho cấu hình đã chọn
Đặt `BEST_FACE` theo bảng val ở trên (tiêu chí: BCubed F1 val cao nhất; xem thêm số cụm so với số nhân vật thật).

In [ ]:
BEST_FACE = "f_a"   # <-- sửa sau khi xem bước 5
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag {BEST_FACE} --unfreeze 4 --eval-only
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag {BEST_FACE} --feat face --emb-dir /kaggle/working/out/reid

## 7. Body (một cấu hình, cùng cách)
Xoá cache face trước để đỡ đầy ổ đĩa (crop body train ~9 GB).

In [ ]:
!rm -rf {CACHE}/face_*
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls body --tag b_a --unfreeze 4 --lr-bb 2e-5 --epochs 10
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag b_a --feat body --emb-dir /kaggle/working/out/reid --no-test | grep -E "feat=|thr|chọn"

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls body --tag b_a --unfreeze 4 --eval-only
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag b_a --feat body --emb-dir /kaggle/working/out/reid

## 8. Lưu kết quả (chỉ /kaggle/working được giữ)
`out/reid/<tag>/best.pt` + `log.json`, và embedding `*.npz`.

In [ ]:
!rm -rf /kaggle/working/m109 /kaggle/working/out/smoke
!cp /kaggle/working/characters.json /kaggle/working/out/
!du -sh /kaggle/working/out/*; ls /kaggle/working/out/reid